# F5 Churn Propensity Model

Build an XGBoost classifier to predict which accounts have declining consumption patterns using telemetry trends, support case patterns, and health scores.

In [ ]:
# Setup and Connection
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import json

# Connect using active Snowflake connection
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set context
session.sql("USE ROLE SYSADMIN").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()
session.sql("USE DATABASE F5_PROD").collect()
session.sql("USE SCHEMA RAW").collect()

# Verify connection
print(f"Connected as: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

## Feature Engineering
Build one row per account combining telemetry signals, trend slopes, support case patterns, and health scores.

In [ ]:
# Feature Engineering - Build CHURN_FEATURES table
session.sql("CREATE SCHEMA IF NOT EXISTS F5_PROD.FINAL").collect()

feature_sql = """
CREATE OR REPLACE TABLE F5_PROD.FINAL.CHURN_FEATURES AS
WITH telemetry_stats AS (
    SELECT SFDCF5_ACCT_ID,
        -- Signal classification
        CASE
            WHEN AVG(BOT_ADVANCED_TRANSACTION_CNT) > 300000 THEN 'bot_defense'
            WHEN AVG(WAF_USAGE_QTY) > 25 THEN 'waf'
            WHEN AVG(ACTIVE_ENDPOINT_QTY) > 150 THEN 'capacity'
            WHEN AVG(ACTIVE_HTTP_LOAD_BALANCER_QTY) > 40 THEN 'load_balancer'
            WHEN AVG(DNS_ZONES_QTY) > 7 THEN 'dns'
            ELSE 'load_balancer'
        END AS DOMINANT_SIGNAL,
        -- Volume averages (60-day)
        AVG(ACTIVE_HTTP_LOAD_BALANCER_QTY) AS AVG_HTTP_LB,
        AVG(WAF_USAGE_QTY) AS AVG_WAF,
        AVG(BOT_ADVANCED_TRANSACTION_CNT) AS AVG_BOT_TXN,
        AVG(ACTIVE_ENDPOINT_QTY) AS AVG_ENDPOINTS,
        AVG(DNS_ZONES_QTY) AS AVG_DNS,
        -- Volume stddev
        STDDEV(ACTIVE_HTTP_LOAD_BALANCER_QTY) AS STDDEV_HTTP_LB,
        STDDEV(WAF_USAGE_QTY) AS STDDEV_WAF,
        STDDEV(BOT_ADVANCED_TRANSACTION_CNT) AS STDDEV_BOT_TXN,
        STDDEV(ACTIVE_ENDPOINT_QTY) AS STDDEV_ENDPOINTS
    FROM F5_PROD.RAW.COL_XC_TELEMETRY
    WHERE OBSERVATION_DATE >= CURRENT_DATE() - 60
    GROUP BY 1
),
telemetry_slopes AS (
    SELECT SFDCF5_ACCT_ID,
        REGR_SLOPE(ACTIVE_HTTP_LOAD_BALANCER_QTY, DATEDIFF(day, '2020-01-01', OBSERVATION_DATE)) AS HTTP_LB_SLOPE,
        REGR_SLOPE(WAF_USAGE_QTY, DATEDIFF(day, '2020-01-01', OBSERVATION_DATE)) AS WAF_SLOPE,
        REGR_SLOPE(BOT_ADVANCED_TRANSACTION_CNT, DATEDIFF(day, '2020-01-01', OBSERVATION_DATE)) AS BOT_SLOPE,
        REGR_SLOPE(ACTIVE_ENDPOINT_QTY, DATEDIFF(day, '2020-01-01', OBSERVATION_DATE)) AS ENDPOINT_SLOPE,
        REGR_SLOPE(DNS_ZONES_QTY, DATEDIFF(day, '2020-01-01', OBSERVATION_DATE)) AS DNS_SLOPE
    FROM F5_PROD.RAW.COL_XC_TELEMETRY
    WHERE OBSERVATION_DATE >= CURRENT_DATE() - 90
    GROUP BY 1
),
support_features AS (
    SELECT d.SFDCF5_ACCT_ID,
        COUNT(CASE WHEN d.SUPPORT_CASE_STATUS_CODE IN ('Open','In Progress') THEN 1 END) AS OPEN_CASE_COUNT,
        COUNT(*) AS TOTAL_CASES_180D,
        AVG(f.TIME_TO_RESOLUTION_MINUTES_NUM) / 60.0 AS AVG_RESOLUTION_HOURS,
        COUNT(CASE WHEN f.TIME_OVER_UNDER_SLA_MINUTES_NUM > 0 THEN 1 END) AS SLA_BREACH_COUNT,
        COUNT(CASE WHEN d.CURRENT_PRIORITY_CODE IN ('P1 - Critical','P2 - High') THEN 1 END) AS ESCALATION_COUNT
    FROM F5_PROD.RAW.DIM_SUPPORT_CASE d
    LEFT JOIN F5_PROD.RAW.FACT_SUPPORT_CASE f ON d.SUPPORT_CASE_ID = f.SUPPORT_CASE_ID
    WHERE d.CREATED_DATETIME >= DATEADD(day, -180, CURRENT_TIMESTAMP())
    GROUP BY 1
),
health AS (
    SELECT SFDCF5_ACCT_ID,
        MAX(SKU_UTILIZATION_PCT) AS SKU_UTILIZATION_PCT,
        MAX(CONSUMPTION_PATTERN) AS CONSUMPTION_PATTERN
    FROM F5_PROD.RAW.COL_XC_PRODUCT_HEALTHSCORE
    GROUP BY 1
)
SELECT 
    ts.SFDCF5_ACCT_ID,
    ts.DOMINANT_SIGNAL,
    ts.AVG_HTTP_LB, ts.AVG_WAF, ts.AVG_BOT_TXN, ts.AVG_ENDPOINTS, ts.AVG_DNS,
    COALESCE(ts.STDDEV_HTTP_LB, 0) AS STDDEV_HTTP_LB,
    COALESCE(ts.STDDEV_WAF, 0) AS STDDEV_WAF,
    COALESCE(ts.STDDEV_BOT_TXN, 0) AS STDDEV_BOT_TXN,
    COALESCE(ts.STDDEV_ENDPOINTS, 0) AS STDDEV_ENDPOINTS,
    COALESCE(sl.HTTP_LB_SLOPE, 0) AS HTTP_LB_SLOPE,
    COALESCE(sl.WAF_SLOPE, 0) AS WAF_SLOPE,
    COALESCE(sl.BOT_SLOPE, 0) AS BOT_SLOPE,
    COALESCE(sl.ENDPOINT_SLOPE, 0) AS ENDPOINT_SLOPE,
    COALESCE(sl.DNS_SLOPE, 0) AS DNS_SLOPE,
    COALESCE(sf.OPEN_CASE_COUNT, 0) AS OPEN_CASE_COUNT,
    COALESCE(sf.TOTAL_CASES_180D, 0) AS TOTAL_CASES_180D,
    COALESCE(sf.AVG_RESOLUTION_HOURS, 0) AS AVG_RESOLUTION_HOURS,
    COALESCE(sf.SLA_BREACH_COUNT, 0) AS SLA_BREACH_COUNT,
    COALESCE(sf.ESCALATION_COUNT, 0) AS ESCALATION_COUNT,
    COALESCE(h.SKU_UTILIZATION_PCT, 0) AS SKU_UTILIZATION_PCT,
    CASE WHEN h.CONSUMPTION_PATTERN = 'Declining' THEN 1 ELSE 0 END AS TARGET
FROM telemetry_stats ts
LEFT JOIN telemetry_slopes sl ON ts.SFDCF5_ACCT_ID = sl.SFDCF5_ACCT_ID
LEFT JOIN support_features sf ON ts.SFDCF5_ACCT_ID = sf.SFDCF5_ACCT_ID
LEFT JOIN health h ON ts.SFDCF5_ACCT_ID = h.SFDCF5_ACCT_ID
"""

session.sql(feature_sql).collect()

# Preview the features
features_df = session.table("F5_PROD.FINAL.CHURN_FEATURES")
print(f"Feature table: {features_df.count()} accounts, {len(features_df.columns)} columns")
print(f"Target distribution:")
features_df.group_by("TARGET").count().show()
print(f"\nSignal distribution:")
features_df.group_by("DOMINANT_SIGNAL").count().show()
print(f"\nSample rows:")
features_df.limit(5).show()

## Feature Store Registration
Register features for centralized management and reuse.

In [ ]:
# Feature Store Registration
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

# Create or connect to Feature Store
fs = FeatureStore(
    session=session,
    database="F5_PROD",
    name="FINAL",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

# Define the account entity
account_entity = Entity(
    name="ACCOUNT",
    join_keys=["SFDCF5_ACCT_ID"],
    desc="F5 customer account for churn prediction"
)
fs.register_entity(account_entity)

# Create feature view from our feature table
feature_df = session.table("F5_PROD.FINAL.CHURN_FEATURES").drop("TARGET")

churn_fv = FeatureView(
    name="FV_CHURN_FEATURES",
    entities=[account_entity],
    feature_df=feature_df,
    desc="Churn propensity features: telemetry signals, trends, support patterns"
)
churn_fv = fs.register_feature_view(churn_fv, version="V1", block=True)

print(f"Feature View registered: {churn_fv.name} (version {churn_fv.version})")
print(f"Features: {len(churn_fv.feature_names)} columns")

# Generate training dataset
spine_df = session.table("F5_PROD.FINAL.CHURN_FEATURES").select("SFDCF5_ACCT_ID", "TARGET").filter(F.col("TARGET").is_not_null())

# Drop existing dataset if re-running
session.sql("DROP TABLE IF EXISTS F5_PROD.FINAL.CHURN_TRAINING_DS_V1").collect()

dataset = fs.generate_dataset(
    name="F5_PROD.FINAL.CHURN_TRAINING_DS",
    spine_df=spine_df,
    features=[churn_fv],
    version="V1",
    output_type="table",
    spine_label_cols=["TARGET"],
    desc="Training dataset for churn propensity model"
)

print(f"\nTraining dataset created: F5_PROD.FINAL.CHURN_TRAINING_DS_V1")
training_df = session.table("F5_PROD.FINAL.CHURN_TRAINING_DS_V1")
print(f"Rows: {training_df.count()}")

## Preprocessing Pipeline
Encode categorical signal feature and scale numeric features. Split into train/test.

In [ ]:
# Preprocessing + Train/Test Split
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.preprocessing import MinMaxScaler, OneHotEncoder

# Load training data
train_data = session.table("F5_PROD.FINAL.CHURN_TRAINING_DS_V1")

# Identify columns
cat_cols = ["DOMINANT_SIGNAL"]
label_col = "TARGET"
exclude_cols = ["SFDCF5_ACCT_ID", label_col]
num_cols = [c for c in train_data.columns if c not in cat_cols + exclude_cols]

print(f"Categorical features: {cat_cols}")
print(f"Numeric features ({len(num_cols)}): {num_cols}")
print(f"Label: {label_col}")

# Build preprocessing pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("OHE", OneHotEncoder(
            input_cols=cat_cols,
            output_cols=cat_cols,
            drop_input_cols=True,
            drop="first",
            handle_unknown="ignore"
        )),
        ("MMS", MinMaxScaler(
            clip=True,
            input_cols=num_cols,
            output_cols=num_cols
        ))
    ]
)

# Fit and transform
processed_df = preprocessing_pipeline.fit(train_data).transform(train_data)

# Train/test split (80/20)
train_df, test_df = processed_df.random_split([0.8, 0.2], seed=42)

print(f"\nTrain set: {train_df.count()} rows")
print(f"Test set: {test_df.count()} rows")
print(f"Total features after encoding: {len([c for c in processed_df.columns if c not in exclude_cols])}")

# Get final feature columns
FEATURE_COLS = [c for c in processed_df.columns if c not in exclude_cols]
print(f"\nFeature columns for model: {FEATURE_COLS}")

## Model Training
Train XGBoost with GridSearch hyperparameter tuning.

In [ ]:
# XGBoost + GridSearch
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.model_selection import GridSearchCV

OUTPUT_COL = "PREDICTED_TARGET"

# Parameter grid for tuning
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5],
}

# GridSearch with 5-fold CV
grid_search = GridSearchCV(
    estimator=XGBClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    input_cols=FEATURE_COLS,
    label_cols=label_col,
    output_cols=OUTPUT_COL
)

print("Training with GridSearch (this may take 1-2 minutes)...")
grid_search.fit(train_df)

# Best parameters
best_model = grid_search.to_sklearn().best_estimator_
print(f"\nBest parameters: {grid_search.to_sklearn().best_params_}")
print(f"Best CV score: {grid_search.to_sklearn().best_score_:.4f}")

## Model Evaluation
Evaluate on held-out test set and visualize feature importance.

In [ ]:
# Evaluation Metrics + Feature Importance
from snowflake.ml.modeling.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
import pandas as pd

# Predict on test set
result = grid_search.predict(test_df)

# Calculate metrics
metrics = {
    "accuracy": accuracy_score(df=result, y_true_col_names="TARGET", y_pred_col_names=OUTPUT_COL),
    "precision": precision_score(df=result, y_true_col_names="TARGET", y_pred_col_names=OUTPUT_COL),
    "recall": recall_score(df=result, y_true_col_names="TARGET", y_pred_col_names=OUTPUT_COL),
    "f1_score": f1_score(df=result, y_true_col_names="TARGET", y_pred_col_names=OUTPUT_COL),
}

print("=" * 50)
print("MODEL EVALUATION METRICS")
print("=" * 50)
for metric, value in metrics.items():
    print(f"  {metric:12s}: {value:.4f}")

cm = confusion_matrix(df=result, y_true_col_name="TARGET", y_pred_col_name=OUTPUT_COL)
print(f"\nConfusion Matrix:")
print(f"  TN={int(cm[0][0]):3d}  FP={int(cm[0][1]):3d}")
print(f"  FN={int(cm[1][0]):3d}  TP={int(cm[1][1]):3d}")

# Feature importance
print("\n" + "=" * 50)
print("TOP 10 FEATURE IMPORTANCE")
print("=" * 50)
importances = best_model.feature_importances_
feat_imp = sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True)[:10]
for name, imp in feat_imp:
    bar = "█" * int(imp * 50)
    print(f"  {name:25s} {imp:.4f} {bar}")

## Model Registry
Log the trained model to the Snowflake Model Registry with metrics.

In [ ]:
# Register in Model Registry
from snowflake.ml.registry import Registry

# Create registry connection
reg = Registry(session=session, database_name="F5_PROD", schema_name="FINAL")

# Sample input for schema
sample_input = train_df.select(FEATURE_COLS).limit(100)

# Log the model
model_name = "CHURN_XGB_MODEL"
version_name = "V1"

mv = reg.log_model(
    model_name=model_name,
    version_name=version_name,
    model=grid_search,
    metrics=metrics,
    sample_input_data=sample_input
)

mv.comment = "XGBoost churn propensity model predicting Declining consumption pattern from telemetry + support features."

print(f"Model registered: {model_name} (version {version_name})")
print(f"Metrics logged: {metrics}")
print(f"\nRegistered models:")
print(reg.show_models())

## Batch Inference
Score ALL accounts with telemetry and write predictions table for downstream use by the agent and Streamlit app.

In [ ]:
# Batch Inference → CHURN_PREDICTIONS
# Load all accounts (not just those with labels)
all_features = session.table("F5_PROD.FINAL.CHURN_FEATURES")

# Preprocess all accounts the same way
all_processed = preprocessing_pipeline.transform(all_features)

# Get predictions
all_predictions = grid_search.predict(all_processed)

# Determine top risk factors from feature importance (global, not per-account)
top_3_factors = ", ".join([f[0] for f in feat_imp[:3]])

# Build predictions - join back to get DOMINANT_SIGNAL and ACCT_NAME
features_ref = session.table("F5_PROD.FINAL.CHURN_FEATURES").select("SFDCF5_ACCT_ID", "DOMINANT_SIGNAL")
acct_df = session.table("F5_PROD.RAW.DIM_CUST_ACCT_SFDC").select("SFDCF5_ACCT_ID", "ACCT_NAME")

predictions_df = all_predictions.select(
    F.col("SFDCF5_ACCT_ID"),
    F.col(OUTPUT_COL).cast("FLOAT").alias("CHURN_RISK_SCORE"),
    F.when(F.col(OUTPUT_COL) == 1, F.lit("At Risk")).otherwise(F.lit("Stable")).alias("PREDICTED_PATTERN"),
    F.lit(top_3_factors).alias("TOP_RISK_FACTORS"),
    F.current_date().alias("PREDICTION_DATE")
)

final_predictions = (
    predictions_df
    .join(features_ref, "SFDCF5_ACCT_ID")
    .join(acct_df, "SFDCF5_ACCT_ID")
    .select("SFDCF5_ACCT_ID", "ACCT_NAME", "CHURN_RISK_SCORE", 
            "PREDICTED_PATTERN", "DOMINANT_SIGNAL", "TOP_RISK_FACTORS", "PREDICTION_DATE")
)

final_predictions.write.mode("overwrite").save_as_table("F5_PROD.FINAL.CHURN_PREDICTIONS")

# Verify
result_table = session.table("F5_PROD.FINAL.CHURN_PREDICTIONS")
print(f"CHURN_PREDICTIONS created: {result_table.count()} accounts")
print(f"\nPrediction distribution:")
result_table.group_by("PREDICTED_PATTERN").count().show()
print(f"\nTop 10 highest risk accounts:")
result_table.filter(F.col("PREDICTED_PATTERN") == "At Risk").order_by("ACCT_NAME").limit(10).show()